# 01 — Clinical Document Parsing & Structuring (from scratch, offline)

Companion notebook to `../01-clinical-document-structuring-and-parsing.md`.

This notebook implements, on a **synthetic** clinical-trial protocol string, the two core
mechanics from that chapter:

1. **Section/heading detection** — regex + numbering-hierarchy parsing (Python `re` only).
2. **Section-boundary-preserving chunking** — splitting long sections into model-context-sized
   pieces without cutting a paragraph (and therefore never an eligibility criterion or dosing
   rule) in half.

Everything here runs fully offline with the Python standard library only — no API keys, no
model downloads, no GPU. This mirrors (at toy scale) the first stage of the ICF/PLPS/SOC
generation pipeline: nothing gets generated until the source Protocol has been reliably broken
into labeled, hierarchical sections.

## 1. A synthetic clinical trial protocol

A short, invented protocol excerpt with the kind of numbered-heading structure real Protocol
templates use: top-level sections (`1`, `2`, `3`, ...) and nested subsections (`2.1`, `3.2`,
...). Eligibility criteria are written as dash-bulleted lists — deliberately *not* numbered —
which sidesteps a real parsing gotcha worth knowing: a numbered criteria list (`1. Adults
aged...`, `2. Confirmed diagnosis...`) can be visually indistinguishable from a heading pattern
using numbering alone. In a production parser you'd disambiguate using style cues (heading
font/bold) or a known-heading vocabulary, as discussed in the chapter; here we sidestep it by
using dash bullets for list items, and note the gotcha explicitly instead of silently avoiding
it.

In [ ]:
SYNTHETIC_PROTOCOL = """1. Background
This study evaluates the safety and efficacy of Drug X in adult patients with moderate
persistent asthma who remain symptomatic despite standard inhaled corticosteroid therapy.
Drug X is a monoclonal antibody targeting interleukin-5 (IL-5).

2. Objectives
2.1 Primary Objective
To evaluate the change from baseline in forced expiratory volume (FEV1) at Week 24
compared to placebo.

2.2 Secondary Objectives
To evaluate the annualized asthma exacerbation rate and change in Asthma Control
Questionnaire (ACQ-6) score at Week 24.

3. Eligibility Criteria
3.1 Inclusion Criteria
- Adults aged 18 to 75 years, inclusive, at the time of screening.
- Confirmed diagnosis of moderate to severe persistent asthma for at least 12 months.
- Able to provide written informed consent prior to any study-related procedures.
- Pre-bronchodilator FEV1 of 40 percent to 90 percent of predicted normal value at screening.

3.2 Exclusion Criteria
- Clinically significant pulmonary disease other than asthma.
- Current smoker or history of smoking exceeding 10 pack-years.
- Use of any investigational drug within 30 days prior to screening.
- Known hypersensitivity to the study drug or any of its excipients.

4. Study Design
This is a randomized, double-blind, placebo-controlled, parallel-group, multicenter study
conducted across approximately 45 sites in 8 countries. Eligible participants will be
randomized in a 1:1 ratio to receive either Drug X or matching placebo, administered
subcutaneously once every 4 weeks for 24 weeks, followed by a 8 week safety follow-up period.

5. Dosing and Administration
5.1 Dose and Regimen
Participants weighing less than 60 kg will receive 100 mg subcutaneously every 4 weeks.
Participants weighing 60 kg or more will receive 200 mg subcutaneously every 4 weeks.
Dose adjustments are not permitted during the treatment period.

5.2 Safety Monitoring
Participants will be monitored for injection site reactions for 30 minutes following each
dose during the first three administrations. Vital signs will be recorded pre-dose and
30 minutes post-dose at every visit.
"""

print(f"Synthetic protocol is {len(SYNTHETIC_PROTOCOL)} characters, "
      f"{len(SYNTHETIC_PROTOCOL.splitlines())} lines.")

## 2. Heading detection with regex

A numbered-heading pattern: one or more dot-separated digits, an optional trailing period,
whitespace, then a capitalized title on the rest of the line. This is the "pattern-based
detection" signal described in the chapter — in a real pipeline it would be combined with
document style cues (bold/font-size) and a known-heading vocabulary, but the regex layer alone
is enough to demonstrate the mechanics.

In [ ]:
import re
from pprint import pprint

HEADING_RE = re.compile(r"^(?P<number>\d+(?:\.\d+)*)\.?\s+(?P<title>[A-Z][^\n]{2,80})$", re.MULTILINE)

def find_headings(text: str):
    """Return heading anchors: number, title, and character offsets in the source text."""
    return [
        {
            "number": m.group("number"),
            "title": m.group("title").strip(),
            "start": m.start(),
            "end_of_heading": m.end(),
        }
        for m in HEADING_RE.finditer(text)
    ]

headings = find_headings(SYNTHETIC_PROTOCOL)
pprint(headings)
print(f"\n{len(headings)} headings found")

## 3. Building hierarchical section objects

A heading number's **depth** (how many dot-separated components it has) tells us where it sits
in the hierarchy: `3` is a top-level section, `3.1` is a subsection of `3`. Slicing the source
text between consecutive heading anchors gives each heading its **body** text, and recording
the parent number lets downstream code ask "give me this section and everything under it"
(exactly what the ICF/PLPS generation prompt for "Eligibility Criteria" needs — the full
inclusion *and* exclusion subsections, not just the top-level heading's own body, which here is
empty since the heading is immediately followed by its first subsection).

In [ ]:
def heading_depth(number: str) -> int:
    return number.count(".") + 1

def build_sections(text: str, headings):
    sections = []
    for i, h in enumerate(headings):
        body_start = h["end_of_heading"]
        body_end = headings[i + 1]["start"] if i + 1 < len(headings) else len(text)
        body = text[body_start:body_end].strip()
        sections.append({
            "number": h["number"],
            "title": h["title"],
            "depth": heading_depth(h["number"]),
            "parent": ".".join(h["number"].split(".")[:-1]) or None,
            "body": body,
        })
    return sections

sections = build_sections(SYNTHETIC_PROTOCOL, headings)

for s in sections:
    indent = "  " * (s["depth"] - 1)
    print(f"{indent}{s['number']} {s['title']}  (parent={s['parent']}, body={len(s['body'])} chars)")

In [ ]:
def get_section_and_children(sections, number_prefix):
    """All sections at `number_prefix` or nested under it -- e.g. "3" -> ["3", "3.1", "3.2"]."""
    return [
        s for s in sections
        if s["number"] == number_prefix or s["number"].startswith(number_prefix + ".")
    ]

eligibility = get_section_and_children(sections, "3")
print("Eligibility Criteria + subsections (what an ICF/PLPS generation prompt would receive):\n")
for s in eligibility:
    print(f"--- {s['number']} {s['title']} ---")
    print(s["body"] if s["body"] else "(heading only, content is in subsections)")
    print()

assert [s["number"] for s in eligibility] == ["3", "3.1", "3.2"]
print("OK: eligibility criteria section + both subsections retrieved together.")

## 4. Section-boundary-preserving chunking

If a section is too long for a single generation call's context budget, the chunker below
splits at **paragraph boundaries only** — never mid-sentence, and never in a way that would
split one bullet of an eligibility-criteria list from the paragraph block it belongs to. This
is deliberately simple (real systems would also fall back to sentence boundaries, and track
real token counts rather than a `chars / 4` approximation) but the core principle is the part
that matters for an interview: **the chunk boundary is chosen by document structure, not by an
arbitrary token count.**

We simulate a section too long for a small context budget by concatenating the Inclusion and
Exclusion Criteria bodies three times over (a stand-in for a much longer real section) and
chunking with a deliberately small `max_tokens`.

In [ ]:
def chunk_section(section_text: str, max_tokens: int, approx_chars_per_token: int = 4):
    """Split section_text into chunks <= max_tokens (approx), breaking only at blank-line
    paragraph boundaries so a criterion / sentence is never split across chunks."""
    max_chars = max_tokens * approx_chars_per_token
    if len(section_text) <= max_chars:
        return [section_text]

    paragraphs = section_text.split("\n\n")
    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) > max_chars and current:
            chunks.append(current.strip())
            current = ""
        current += para + "\n\n"
    if current.strip():
        chunks.append(current.strip())
    return chunks

inclusion = next(s for s in sections if s["number"] == "3.1")
exclusion = next(s for s in sections if s["number"] == "3.2")
long_criteria_text = (inclusion["body"] + "\n\n" + exclusion["body"] + "\n\n") * 3

chunks = chunk_section(long_criteria_text, max_tokens=60)
print(f"Chunked an artificially-lengthened Eligibility Criteria block into {len(chunks)} chunks "
      f"(max_tokens=60, ~{60*4} chars/chunk budget)\n")
for i, c in enumerate(chunks):
    preview = c[:90].replace(chr(10), " | ")
    print(f"chunk {i}: {len(c)} chars -- {preview}...")

assert len(chunks) > 1, "expected the long block to require multiple chunks"
assert all(len(c) <= 60 * 4 + 400 for c in chunks), "sanity check on chunk sizing"
print("\nOK: every chunk boundary falls on a paragraph break, never mid-bullet.")

## 5. Why this matters (tying back to Chapter 01)

- **Heading detection** turns an unstructured blob of Protocol text into addressable,
  hierarchical sections — this is what lets a downstream generation call ask for exactly
  "Eligibility Criteria, all subsections" instead of guessing at offsets in a flat document.
- **Boundary-preserving chunking** guarantees that even when a section has to be split to fit a
  model's context window, no individual criterion, dosing rule, or safety instruction is ever
  silently cut in half — which is exactly the kind of silent corruption that would be dangerous
  to ship in an ICF or PLPS section (a missing exclusion criterion is a patient-safety issue,
  not just an inconvenience).
- **A real pipeline goes further**: table extraction with row/column structure preserved
  (Chapter 01), style-cue and known-vocabulary heading detection to disambiguate cases the pure
  regex approach here would get wrong (like numbered eligibility criteria that look like
  headings), and post-processing/terminology normalization once generation has run. This
  notebook demonstrates the mechanics at toy scale; the chapter covers the full picture.